## GPU
使用nvidia-smi查看显卡信息

In [1]:
!nvidia-smi

Sat Feb 24 21:07:08 2024       
+---------------------------------------------------------------------------------------+
| NVIDIA-SMI 546.29                 Driver Version: 546.29       CUDA Version: 12.3     |
|-----------------------------------------+----------------------+----------------------+
| GPU  Name                     TCC/WDDM  | Bus-Id        Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |         Memory-Usage | GPU-Util  Compute M. |
|                                         |                      |               MIG M. |
|=========================================+======================+======================|
|   0  NVIDIA GeForce RTX 4060 Ti   WDDM  | 00000000:01:00.0  On |                  N/A |
|  0%   28C    P8              13W / 165W |    903MiB / 16380MiB |     29%      Default |
|                                         |                      |                  N/A |
+-----------------------------------------+----------------------+--

### 计算设备
我们可以指定用于存储和计算的设备，如CPU和GPU。 默认情况下，张量是在内存中创建的，然后使用CPU计算它。
在PyTorch中，CPU和GPU可以用torch.device('cpu') 和torch.device('cuda')表示。 应该注意的是，cpu设备意味着所有物理CPU和内存， 这意味着PyTorch的计算将尝试使用所有CPU核心。 然而，gpu设备只代表一个卡和相应的显存。 如果有多个GPU，我们使用torch.device(f'cuda:{i}') 来表示第i块GPU（i从0开始）。 另外，cuda:0和cuda是等价的。

In [2]:
import torch    
from torch import nn

(torch.device('cpu'),torch.device('cuda'),torch.device(type='cuda',index=1))

(device(type='cpu'), device(type='cuda'), device(type='cuda', index=1))

In [4]:
# 查询可用gpu数量
torch.cuda.device_count()
# 定义了两个方便的函数， 这两个函数允许我们在不存在所需所有GPU的情况下运行代码。
def try_gpu(i=0):
    """如果存在，则返回gpu(i),否则返回cpu()"""
    if torch.cuda.device_count() >= i + 1:
        return torch.device(f'cuda:{i}')
    return torch.device('cpu')

def try_all_gpus():
    """返回所有可用gpu，如果没有gpu，则返回[cpu(),]"""  
    devices = [torch.device(f'cuda:{i}') for i in range(torch.cuda.device_count())]
    return devices if devices else  [torch.device('cpu')]

try_gpu(),try_gpu(10),try_all_gpus()

(device(type='cuda', index=0),
 device(type='cpu'),
 [device(type='cuda', index=0)])

### 张量与GPU


In [5]:
x = torch.tensor([1,2,3])
x.device

device(type='cpu')

#### 存储在GPU上
有几种方法可以在GPU上存储张量。 例如，我们可以在创建张量时指定存储设备。接 下来，我们在第一个gpu上创建张量变量X。 在GPU上创建的张量只消耗这个GPU的显存。

In [7]:
x = torch.ones(2,3,device=try_gpu())
x

tensor([[1., 1., 1.],
        [1., 1., 1.]], device='cuda:0')

#### 复制
如果我们要计算X + Y，我们需要决定在哪里执行这个操作。 我们可以将X传输到第二个GPU并在那里执行操作。 不要简单地X加上Y，因为这会导致异常， 运行时引擎不知道该怎么做：它在同一设备上找不到数据会导致失败。 由于Y位于第二个GPU上，所以我们需要将X移到那里， 然后才能执行相加运算。
### 神经网络与GPU


In [10]:
net = nn.Sequential(nn.Linear(3,1)) 
net = net.to(device=try_gpu())
net(x)

tensor([[1.0554],
        [1.0554]], device='cuda:0', grad_fn=<AddmmBackward0>)

In [11]:
# 让我们确认模型参数存储在同一个gpu上
net[0].weight.data.device   

device(type='cuda', index=0)